# HIPO — آزمایش قابل ممیزی روی تیک واقعی

**هدف پژوهش:** متوسط سود خالص ماهانه ۳–۴٪ و افت سرمایه اکوییتی زیر ۱۰٪ روی یک جفت ارز. این نسخه **استراتژی موفق اثبات‌شده نیست**؛ داده در دسترس این اجرا یک هفته است و هدف را تأیید نمی‌کند. افزایش اهرم برای نمایش سود هدف ممنوع است.

نوت‌بوک خودکفا است؛ فایل Python جدا یا clone لازم ندارد. حالت پیش‌فرض `sample` برای بررسی اجراست. برای پژوهش چندساله از `histdata` یا `local` استفاده کنید، پارامترها را **پیش از دیدن نتیجه** تثبیت کنید و همه سلول‌ها را به ترتیب اجرا کنید.

[مخزن GitHub](https://github.com/khajavy8056/Hippoalgorithm-) · داده نمونه: آینه عمومی GAIN Capital، GBPUSD، هفته اول آوریل ۲۰۱۶؛ صحت منبع مستقلاً توسط بروکر تأیید نشده است.


In [ ]:
# @title ۱ — دریافت تیک واقعی (همان روش token + POST فایل ارسالی)
import sys, subprocess, importlib.util, json, glob, hashlib
from pathlib import Path
requirements = ['numpy>=2.2,<2.5', 'pandas>=2.2,<3.1', 'pyarrow>=18,<26',
                'numba>=0.61,<0.69', 'scikit-learn>=1.6,<1.10', 'matplotlib>=3.9,<3.12',
                'requests>=2.32,<3', 'beautifulsoup4>=4.12,<5']
modules = ['numpy','pandas','pyarrow','numba','sklearn','matplotlib','requests','bs4']
if any(importlib.util.find_spec(m) is None for m in modules):
    subprocess.check_call([sys.executable,'-m','pip','install',*requirements])
import numpy as np, pandas as pd
from IPython.display import display, HTML
PACKAGE = Path('.hippo_notebook') / 'hippo_lab'
PACKAGE.mkdir(parents=True, exist_ok=True)
(PACKAGE / '__init__.py').write_text('')
sys.path.insert(0, str(PACKAGE.parent.resolve()))
# Re-running all cells reloads the embedded implementation, not stale imports.
for name in list(sys.modules):
    if name == 'hippo_lab' or name.startswith('hippo_lab.'):
        del sys.modules[name]
(PACKAGE / 'data.py').write_text('"""Quote loaders. No price-only fallback and no synthetic quotes."""\nfrom pathlib import Path\nfrom datetime import timezone, timedelta\nimport hashlib, io, zipfile, time\nimport numpy as np\nimport pandas as pd\nimport requests\nfrom bs4 import BeautifulSoup\n\nHISTDATA_TZ = timezone(timedelta(hours=-5))  # fixed EST, NOT America/New_York\nSAMPLE_COMMIT = \'1e0879d8214bafffe44d48e61d1ec6410ec4e7ec\'\n\n\ndef validate_ticks(frame, naive_tz=None):\n    f = frame.rename(columns={c: str(c).lower() for c in frame.columns}).copy()\n    if \'timestamp\' in f:\n        f.index = pd.DatetimeIndex(pd.to_datetime(f.pop(\'timestamp\'), format=\'mixed\'))\n    if not isinstance(f.index, pd.DatetimeIndex):\n        raise ValueError(\'DatetimeIndex or timestamp column required\')\n    if f.index.tz is None:\n        if naive_tz is None:\n            raise ValueError(\'Naive timestamps: explicitly provide source timezone\')\n        f.index = f.index.tz_localize(naive_tz, ambiguous=\'raise\', nonexistent=\'raise\')\n    f.index = f.index.tz_convert(\'UTC\').as_unit(\'ns\')\n    if not {\'bid\', \'ask\'} <= set(f):\n        raise ValueError(\'Real bid AND ask required. Price-only data are not executable quotes.\')\n    f = f[[\'bid\', \'ask\']].astype(float)\n    good = np.isfinite(f).all(axis=1) & (f.bid > 0) & (f.ask >= f.bid)\n    audit = {\'input_rows\': len(f), \'invalid_rows\': int((~good).sum()),\n             \'out_of_order_rows\': int((f.index.to_series().diff().dt.total_seconds() < 0).sum()),\n             \'duplicate_timestamps\': int(f.index.duplicated().sum())}\n    # Preserve differing quotes with equal timestamps, and their original source order.\n    f = f.loc[good].sort_index(kind=\'stable\')\n    if len(f) < 2:\n        raise ValueError(\'Insufficient valid quotes\')\n    gaps = f.index.to_series().diff().dt.total_seconds()\n    audit.update(valid_rows=len(f), start=str(f.index[0]), end=str(f.index[-1]),\n                 gaps_over_60s=int((gaps > 60).sum()), gaps_over_5min=int((gaps > 300).sum()), max_gap_seconds=float(gaps.max()),\n                 median_spread=float((f.ask-f.bid).median()),\n                 p99_spread=float((f.ask-f.bid).quantile(.99)))\n    return f, audit\n\n\ndef read_ticks(path, naive_tz=HISTDATA_TZ):\n    p = Path(path)\n    if p.suffix == \'.parquet\':\n        return validate_ticks(pd.read_parquet(p), naive_tz)\n    if p.suffix == \'.zip\':\n        with zipfile.ZipFile(p) as z:\n            names = [n for n in z.namelist() if n.lower().endswith(\'.csv\')]\n            if len(names) != 1:\n                raise ValueError(\'Choose a ZIP containing exactly one quote CSV\')\n            with z.open(names[0]) as stream:\n                f = pd.read_csv(stream, header=None, names=[\'timestamp\',\'bid\',\'ask\',\'volume\'])\n        f.timestamp = pd.to_datetime(f.timestamp, format=\'%Y%m%d %H%M%S%f\')\n        return validate_ticks(f, naive_tz)\n    return validate_ticks(pd.read_csv(p), naive_tz)\n\n\ndef download_histdata(pair, year, month, folder=\'data/histdata\', retries=3):\n    """Same form-token + POST method as the supplied Colab, fail loudly."""\n    pair = pair.replace(\'/\', \'\').upper()\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\n    dest = folder / f\'{pair}_{year}_{month:02d}.zip\'\n    if dest.exists():\n        read_ticks(dest)  # never silently trust a partial archive\n        return dest\n    page = f\'https://www.histdata.com/download-free-forex-historical-data/?/ascii/tick-data-quotes/{pair.lower()}/{year}/{month}\'\n    errors = []\n    for attempt in range(retries):\n        try:\n            with requests.Session() as s:\n                s.headers.update({\'User-Agent\':\'Mozilla/5.0\', \'Referer\':page})\n                r = s.get(page, timeout=45); r.raise_for_status()\n                form = BeautifulSoup(r.content, \'html.parser\').find(\'form\', id=\'file_down\')\n                if form is None: raise ValueError(\'HistData token form not found\')\n                payload = {t.get(\'name\'): t.get(\'value\',\'\') for t in form.find_all(\'input\') if t.get(\'name\')}\n                r = s.post(\'https://www.histdata.com/get.php\', data=payload, timeout=120)\n                r.raise_for_status()\n                if not zipfile.is_zipfile(io.BytesIO(r.content)): raise ValueError(\'Response is not a ZIP\')\n                tmp = dest.with_suffix(\'.partial.zip\'); tmp.write_bytes(r.content)\n                read_ticks(tmp); tmp.replace(dest)\n                return dest\n        except (requests.RequestException, ValueError, zipfile.BadZipFile) as e:\n            errors.append(str(e)); time.sleep(2**attempt)\n    raise RuntimeError(f\'Failed {pair} {year}-{month:02d}: {errors}; upload original quote ZIP/parquet instead\')\n\n\ndef download_sample(folder=\'data/cache\'):\n    """A public GAIN Capital mirror: provenance claimed by repo, not broker-certified."""\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\n    p = folder/\'gain_sample.csv\'\n    if not p.exists():\n        url = f\'https://codeload.github.com/phuocidi/cleaned_tick_data/zip/{SAMPLE_COMMIT}\'\n        r = requests.get(url, timeout=90); r.raise_for_status()\n        with zipfile.ZipFile(io.BytesIO(r.content)) as z:\n            n = next(n for n in z.namelist() if n.endswith(\'2016_04_GBP_USD_Week1.csv\'))\n            p.write_bytes(z.read(n))\n    f = pd.read_csv(p, header=None, names=[\'pair\',\'timestamp\',\'bid\',\'ask\'])\n    # GAIN historical time is New York local (DST applies); verify against broker before deployment.\n    ticks, audit = validate_ticks(f, \'America/New_York\')\n    audit.update(source=\'Public mirror of GAIN Capital GBPUSD Week1 April 2016\',\n                 commit=SAMPLE_COMMIT, sha256=hashlib.sha256(p.read_bytes()).hexdigest(),\n                 timezone_assumption=\'America/New_York; not independently broker-certified\')\n    return ticks, audit\n\n\ndef make_bars(ticks, rule=\'15min\'):\n    mid = (ticks.bid + ticks.ask)/2\n    bars = mid.resample(rule, label=\'right\', closed=\'left\').ohlc()\n    bars[\'count\'] = mid.resample(rule, label=\'right\', closed=\'left\').count()\n    bars[\'spread\'] = (ticks.ask-ticks.bid).resample(rule, label=\'right\', closed=\'left\').median()\n    bars = bars.dropna()\n    # Do not treat the final unfinished candle as closed.\n    return bars.loc[bars.index <= ticks.index[-1]]\n', encoding='utf-8')

from hippo_lab.data import download_histdata, download_sample, read_ticks, make_bars, validate_ticks, HISTDATA_TZ
DATA_MODE = 'sample' # @param ['sample', 'histdata', 'local']
PAIR = 'GBPUSD' # @param {type:'string'}
ACCOUNT_CURRENCY = 'USD' # @param {type:'string'}
START_YEAR = 2022 # @param {type:'integer'}
END_YEAR = 2024 # @param {type:'integer'}
LOCAL_GLOB = '/content/drive/MyDrive/HIPO_DATA/GBPUSD_TICK_*.parquet' # @param {type:'string'}
MOUNT_DRIVE = False # @param {type:'boolean'}
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if DATA_MODE == 'sample':
    if PAIR != 'GBPUSD': raise ValueError('The pinned sample is GBPUSD only')
    ticks, data_audit = download_sample()
else:
    files = []
    if DATA_MODE == 'histdata':
        now = pd.Timestamp.now(tz='UTC')
        for year in range(START_YEAR, END_YEAR+1):
            for month in range(1,13):
                if pd.Timestamp(year=year,month=month,day=1,tz='UTC')+pd.DateOffset(months=1)>now:
                    continue # skip incomplete current month
                print(f'Downloading {PAIR} {year}-{month:02d}', flush=True)
                files.append(download_histdata(PAIR,year,month))
    elif DATA_MODE == 'local':
        files = sorted(Path(p) for p in glob.glob(LOCAL_GLOB))
    if not files: raise ValueError('No quote files. Mount Drive or correct LOCAL_GLOB.')
    frames=[]; audits=[]; manifest=[]
    for path in files:
        f,a=read_ticks(path, naive_tz=HISTDATA_TZ)
        # Annual files from original downloader use Bid/Ask and fixed EST naive index.
        frames.append(f); audits.append(a)
        h=hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda:stream.read(1024*1024),b''): h.update(block)
        manifest.append({'file':str(path),'sha256':h.hexdigest()})
    ticks, data_audit=validate_ticks(pd.concat(frames))
    del frames
    # Overlapping files are dangerous: equal timestamps with different quotes may be legitimate
    # inside one file, but cross-file overlaps must be corrected by the user.
    ranges=sorted((pd.Timestamp(a['start']),pd.Timestamp(a['end'])) for a in audits)
    if any(b[0]<=a[1] for a,b in zip(ranges,ranges[1:])):
        raise ValueError('Overlapping quote files; choose non-overlapping monthly OR annual files, not both')
    data_audit.update(source='HistData quote ZIP / user original parquet',pair=PAIR,
                      file_audits=audits, manifest=manifest, timezone='HistData fixed EST UTC-5')
if len(ticks)>100_000_000:
    print('WARNING: >100M quotes. High-RAM Colab required; keep only one pair. No downsampling of execution quotes.')
bars = make_bars(ticks, '15min')
print(json.dumps(data_audit, indent=2, ensure_ascii=False))
print('M15 closed bars:',len(bars),'Quote RAM MB:',round(ticks.memory_usage(deep=True).sum()/1e6,1))
display(ticks.head())


## ۲ — قواعد اجرای واقعی‌تر و محدودیت‌ها

- خرید در **Ask** و فروش در **Bid**؛ سیگنال فقط با کندل بسته‌شده، ورود روی تیک بعدی پس از تأخیر.
- کمیسیون هر سمت، لغزش نامطلوب، اسپرد مشاهده‌شده، حد اهرم و ریسک هر معامله ۰٫۲۵٪.
- استاپ در گپ با قیمت مشاهده‌شده پر می‌شود؛ سود حدی با عبور قیمت و بدون بهبود قیمت پر می‌شود. دفتر سفارش و عمق بازار در این داده نیست.
- اکوییتی شامل سود/زیان باز و هزینه خروج در **هر تیک**؛ توقف افت سرمایه ۸٪ و زیان روزانه ۲٪، **بدون تضمین سقف** در گپ.
- فقط یک پوزیشن؛ خروج پیش از rollover نیویورک، بدون فرض سوآپ صفر برای پوزیشن شبانه. اگر داده قطع شود خروج در اولین تیک بعدی است و علامت گپ ثبت می‌شود.
- حساب باید ارز مظنه باشد؛ مثال GBPUSD با USD. این نسخه تبدیل ارز برای حساب USD روی USDJPY یا طلا را پیاده نمی‌کند.
- ساعت HistData، EST ثابت UTC−5 است (بدون DST). فرض ساعت نمونه GAIN، نیویورک است و در گزارش ثبت می‌شود.


In [ ]:
# @title ۲ — موتور تیک‌به‌تیک و تنظیم ریسک
(PACKAGE / 'engine.py').write_text('"""Single-position event-driven quote replay; account currency = pair quote currency."""\nfrom dataclasses import dataclass, asdict\nimport numpy as np\nimport pandas as pd\nfrom numba import njit\n\n@dataclass(frozen=True)\nclass Execution:\n    pair: str = "GBPUSD"\n    account_currency: str = "USD"\n    initial_cash: float = 10000.0\n    risk_fraction: float = .0025\n    max_leverage: float = 5.0\n    slippage: float = .00002  # absolute quote-price units per fill\n    commission_per_million: float = 35.0  # per SIDE, quote currency / million BASE units\n    spread_multiplier: float = 1.0\n    tp_trade_through: float = .00001  # require executable quote beyond limit\n    max_spread: float = .00030\n    latency_ms: int = 250\n    max_entry_delay_s: int = 60\n    max_hold_hours: float = 6.0\n    drawdown_breaker: float = .08\n    daily_loss_limit: float = .02\n    session_start_hour: int = 7\n    session_end_hour: int = 16\n    flatten_hour: float = 16.75  # NY, before rollover; no overnight swap model\n\n@njit\ndef _replay(ts, bid0, ask0, local_day, hour, signal_tick, sides, distances, rr, absolute_stops,\n            cash0, risk, leverage, slip, fee, spread_mult, max_spread,\n            hold_ns, breaker, daily_limit, tp_through, session_start, session_end, flatten_hour):\n    n = len(ts); eq = np.empty(n); ledger = np.empty((len(sides)+1, 9))\n    cash = cash0; peak = cash0; day_cash = cash0; day = local_day[0]\n    pos = 0; units = 0.; entry = 0.; stop = 0.; target = 0.; entry_i = 0\n    ptr = 0; trades = 0; halted = False; day_halt = False; max_dd = 0.\n    for i in range(n):\n        spread = (ask0[i]-bid0[i])*spread_mult\n        mid = (ask0[i]+bid0[i])/2\n        bid = mid-spread/2; ask = mid+spread/2\n        liquidation = bid if pos == 1 else ask\n        mark = cash + pos*units*(liquidation-entry) - (units*fee if pos else 0.)\n        if local_day[i] != day:\n            day = local_day[i]; day_cash = mark; day_halt = False\n        peak = max(peak, mark); dd = max(0., 1-mark/peak); max_dd = max(max_dd, dd)\n        if dd >= breaker: halted = True\n        if mark <= day_cash*(1-daily_limit): day_halt = True\n        exited = False\n        if pos:\n            reason = 0\n            if (pos == 1 and bid <= stop) or (pos == -1 and ask >= stop): reason = 1\n            elif (pos == 1 and bid >= target+tp_through) or (pos == -1 and ask <= target-tp_through): reason = 2\n            elif ts[i]-ts[entry_i] >= hold_ns: reason = 3\n            elif hour[i] >= flatten_hour or local_day[i] != local_day[entry_i]: reason = 4\n            if halted or day_halt: reason = 5\n            if i == n-1: reason = 6\n            if reason:\n                # Stop gaps filled at the observed executable quote, not at requested stop.\n                # Take-profit capped at its limit; no favourable price improvement assumed.\n                fill = liquidation - pos*slip\n                if reason == 2: fill = min(fill, target) if pos == 1 else max(fill, target)\n                pnl = pos*units*(fill-entry) - 2*units*fee\n                cash += pos*units*(fill-entry) - units*fee\n                ledger[trades] = np.array([entry_i,i,pos,units,entry,fill,pnl,reason,stop])\n                trades += 1; pos = 0; exited = True; mark = cash\n        # Signals arriving while busy are discarded, NOT queued for a later entry.\n        while ptr < len(sides) and signal_tick[ptr] <= i:\n            if signal_tick[ptr] == i and not pos and not exited and not halted and not day_halt and i < n-1:\n                d = distances[ptr]; side = sides[ptr]\n                if np.isfinite(absolute_stops[ptr]):\n                    quoted_entry=(ask if side==1 else bid)+side*slip\n                    d=side*(quoted_entry-absolute_stops[ptr])\n                if d > spread+2*slip and spread <= max_spread and session_start <= hour[i] < min(session_end,flatten_hour):\n                    price = (ask if side == 1 else bid) + side*slip\n                    # Include expected stop exit slippage and both commissions in risk budget.\n                    qty = min(cash*risk/(d+slip+2*fee), cash*leverage/price)\n                    if qty > 0:\n                        pos = side; units = qty; entry = price; entry_i = i\n                        stop = absolute_stops[ptr] if np.isfinite(absolute_stops[ptr]) else price-side*d; target = price+side*d*rr[ptr]\n                        cash -= units*fee\n                        mark = cash + pos*units*((bid if pos == 1 else ask)-entry)-units*fee\n            ptr += 1\n        eq[i] = mark\n        peak = max(peak, mark); max_dd = max(max_dd, max(0., 1-mark/peak))\n    return eq, ledger[:trades], max_dd, halted\n\n\ndef backtest(ticks, signals, cfg=Execution()):\n    if len(cfg.pair) != 6 or cfg.account_currency != cfg.pair[-3:]:\n        raise ValueError(\'Account currency must equal quote currency; cross-currency conversion is not implemented\')\n    if cfg.initial_cash <= 0 or not 0 < cfg.risk_fraction < 1 or cfg.slippage < 0 or cfg.spread_multiplier < 1:\n        raise ValueError(\'Invalid execution configuration\')\n    if not ticks.index.is_monotonic_increasing or ticks.index.tz is None:\n        raise ValueError(\'Sorted timezone-aware quotes required\')\n    if (ticks.ask < ticks.bid).any(): raise ValueError(\'Crossed quotes\')\n    if len(ticks) < 2: raise ValueError(\'At least two quotes required\')\n    ts = ticks.index.as_unit(\'ns\').asi8\n    s = signals.sort_index(kind=\'stable\')\n    if len(s) and (not s.index.is_unique or not s.side.isin([-1,1]).all() or (s.distance <= 0).any() or (s.rr <= 0).any()):\n        raise ValueError(\'Signals must have unique times, side +/-1 and positive distance/rr\')\n    ready = s.index.as_unit(\'ns\').asi8 + cfg.latency_ms*1_000_000\n    # Strictly later timestamp even at zero configured latency.\n    inds = np.maximum(np.searchsorted(ts, ready, side=\'left\'), np.searchsorted(ts, s.index.as_unit(\'ns\').asi8, side=\'right\'))\n    valid = inds < len(ts)\n    safe = np.minimum(inds, len(ts)-1)\n    valid &= ts[safe]-ready <= cfg.max_entry_delay_s*1_000_000_000\n    local = ticks.index.tz_convert(\'America/New_York\')\n    days = np.asarray(local.year*10000+local.month*100+local.day, dtype=np.int64)\n    hours = np.asarray(local.hour+local.minute/60+local.second/3600, dtype=float)\n    equity, ledger, max_dd, halted = _replay(ts, ticks.bid.to_numpy(), ticks.ask.to_numpy(), days, hours,\n        inds[valid], s.side.to_numpy(dtype=np.int64)[valid], s.distance.to_numpy(dtype=float)[valid], s.rr.to_numpy(dtype=float)[valid],\n        s.stop_price.to_numpy(dtype=float)[valid] if \'stop_price\' in s else np.full(int(valid.sum()),np.nan),\n        cfg.initial_cash, cfg.risk_fraction, cfg.max_leverage, cfg.slippage, cfg.commission_per_million/1e6,\n        cfg.spread_multiplier, cfg.max_spread, int(cfg.max_hold_hours*3600e9), cfg.drawdown_breaker,\n        cfg.daily_loss_limit, cfg.tp_trade_through, cfg.session_start_hour, cfg.session_end_hour, cfg.flatten_hour)\n    trades = pd.DataFrame(ledger, columns=[\'entry_tick\',\'exit_tick\',\'side\',\'units\',\'entry\',\'exit\',\'pnl\',\'reason\',\'stop\'])\n    if len(trades):\n        gap_seconds=np.r_[0.,np.diff(ts)/1e9]\n        gap_prefix=np.cumsum(gap_seconds>60)\n        trades[\'quote_gap_over_60s\']=gap_prefix[trades.exit_tick.astype(int)]>gap_prefix[trades.entry_tick.astype(int)]\n        trades[\'planned_loss\'] = trades.units*((trades.entry-trades.stop).abs()+cfg.slippage+2*cfg.commission_per_million/1e6)\n        trades[\'realized_R\'] = trades.pnl/trades.planned_loss\n        trades[\'entry_time\'] = ticks.index[trades.entry_tick.astype(int)]\n        trades[\'exit_time\'] = ticks.index[trades.exit_tick.astype(int)]\n    return {\'equity\':pd.Series(equity,index=ticks.index,name=\'equity\'), \'trades\':trades,\n            \'max_drawdown\':float(max_dd), \'halted\':bool(halted), \'config\':asdict(cfg),\n            \'submitted_signals\':len(s), \'stale_or_outside_signals\':int((~valid).sum())}\n', encoding='utf-8')

from hippo_lab.engine import Execution, backtest
execution = Execution(pair=PAIR, account_currency=ACCOUNT_CURRENCY)
# تغییر هزینه‌ها فقط قبل از مشاهده آزمون نهایی؛ برای EURUSD/JPY واحد قیمت لغزش را درست تنظیم کنید.
print(execution)


## ۳ — فرضیه‌ها و پروتکل ضد بیش‌برازش

دو شکست کانال، دو بازگشت به میانگین در رژیم کم‌روند، و دو آستانه برای Logistic Regression با مقیاس‌بندی فقط روی آموزش. مدل عمداً ساده است: پیچیدگی بیشتر الزاماً مزیت معاملاتی نیست. برچسب ML جهت بازده ۸ کندل بعد است، نه سود قابل معامله؛ **معیار انتخاب بازده خالص بک‌تست است نه AUC**.

آموزش ML با برچسب‌های غیرهم‌پوشان، پایان برچسب پیش از مرز و embargo زمانی ۲۴ ساعت. داده تست برای early stopping یا تغییر آستانه مصرف نمی‌شود. ویژگی‌ها فقط گذشته و کندل فعلی بسته‌شده؛ هیچ pivot با اطلاع آینده نداریم.

**حالت کامل:** حداقل ۲۴ ماه کامل ورودی؛ پنجره ۱۲ ماه آموزش + ۳ ماه اعتبارسنجی + ۳ ماه آزمون جلو‌رونده، ۶ ماه آخر مستقل و دست‌نخورده. برای حداقل ۴ fold جلو‌رونده حدود ۳۳ ماه ورودی لازم است؛ برای ۲۴ ماه OOS ورودی بیشتری لازم است. نمونه یک‌هفته‌ای فقط smoke test است و ML عمداً با شواهد ناکافی غیرفعال می‌شود.

گزینه با کمتر از ۳۰ معامله اعتبارسنجی، بازده خالص غیرمثبت، یا DD بالای ۸٪ رد می‌شود؛ در نبود گزینه، **CASH / عدم معامله**. هیچ بازتنظیمی پس از نتیجه holdout انجام نمی‌شود. انتخاب و ضرایب مدل قبل از اجرای holdout در `artifacts/frozen_selection.json` با هش ذخیره می‌شوند.


In [ ]:
# @title ۳ — ویژگی‌ها، مدل و انتخاب زمانی
(PACKAGE / 'research.py').write_text('"""Predeclared hypotheses, time-purged ML and nested chronological selection."""\nfrom dataclasses import replace\nimport hashlib, json\nimport numpy as np\nimport pandas as pd\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom .engine import Execution, backtest\n\nCANDIDATES = [\n    {\'name\':\'breakout_32\', \'family\':\'breakout\', \'lookback\':32, \'stop_atr\':2., \'rr\':1.5},\n    {\'name\':\'breakout_64\', \'family\':\'breakout\', \'lookback\':64, \'stop_atr\':2.5, \'rr\':2.},\n    {\'name\':\'reversion_32\', \'family\':\'reversion\', \'lookback\':32, \'stop_atr\':2., \'rr\':1.},\n    {\'name\':\'reversion_64\', \'family\':\'reversion\', \'lookback\':64, \'stop_atr\':2.5, \'rr\':1.},\n    {\'name\':\'logistic_060\', \'family\':\'ml\', \'threshold\':.60, \'stop_atr\':2., \'rr\':1.5},\n    {\'name\':\'logistic_065\', \'family\':\'ml\', \'threshold\':.65, \'stop_atr\':2., \'rr\':1.5},\n]\nFEATURES = [\'ret1\',\'ret4\',\'ret16\',\'z32\',\'trend\',\'vol\',\'cost\']\nLABEL_HORIZON = 8  # 2 hours on M15\nWARMUP_BARS = 96\nPROTOCOL = {\'candidate_trials\':len(CANDIDATES), \'bar_rule\':\'15min\', \'label_horizon\':LABEL_HORIZON,\n    \'minimum_validation_trades\':30, \'minimum_ml_independent_labels\':100,\n    \'minimum_months_for_target\':24, \'embargo_hours\':24, \'minimum_forward_folds\':4, \'holdout_months\':6, \'selection\':\'positive net return, DD < 8%, maximize net return / max(DD, 0.5%)\',\n    \'no_passing_candidate\':\'CASH\', \'seed\':8056}\n\n\ndef protocol_hash(cfg):\n    from dataclasses import asdict\n    return hashlib.sha256(json.dumps({\'protocol\':PROTOCOL, \'candidates\':CANDIDATES,\n                                      \'execution\':asdict(cfg)},sort_keys=True).encode()).hexdigest()\n\n\ndef features(b):\n    c = b.close; out = pd.DataFrame(index=b.index)\n    tr = pd.concat([b.high-b.low,(b.high-c.shift()).abs(),(b.low-c.shift()).abs()],axis=1).max(axis=1)\n    out[\'atr\'] = tr.rolling(32,min_periods=32).mean()\n    for k in [1,4,16]: out[f\'ret{k}\'] = c.pct_change(k, fill_method=None)\n    out[\'z32\'] = (c-c.rolling(32).mean())/c.rolling(32).std()\n    out[\'trend\'] = (c.rolling(16).mean()-c.rolling(64).mean())/out.atr\n    out[\'vol\'] = out.ret1.rolling(32).std()\n    out[\'cost\'] = b.spread/out.atr\n    return out.replace([np.inf,-np.inf],np.nan)\n\n\ndef fit_candidate(candidate, bars, train_start, train_end):\n    if candidate[\'family\'] != \'ml\': return None\n    x = features(bars)\n    future = bars.close.shift(-LABEL_HORIZON)/bars.close-1\n    label_end = pd.Series(bars.index,index=bars.index).shift(-LABEL_HORIZON)\n    # Earliest feature windows entirely inside train; labels end strictly before boundary.\n    cutoff=train_end-pd.Timedelta(hours=PROTOCOL[\'embargo_hours\'])\n    train = (bars.index >= train_start) & (bars.index < cutoff)\n    eligible = train & (label_end < cutoff) & x[FEATURES].notna().all(axis=1)\n    eligible[:np.searchsorted(bars.index, train_start)+WARMUP_BARS] = False\n    loc = np.flatnonzero(eligible)\n    # Non-overlapping labels for training; effective sample count logged/guarded.\n    loc = loc[::LABEL_HORIZON]\n    if len(loc) < PROTOCOL[\'minimum_ml_independent_labels\'] or (future.iloc[loc] > 0).nunique() < 2:\n        return \'INSUFFICIENT\'\n    model = make_pipeline(StandardScaler(), LogisticRegression(C=.1, max_iter=500, random_state=8056))\n    model.fit(x[FEATURES].iloc[loc], (future.iloc[loc]>0).astype(int))\n    return model\n\n\ndef signals_for(candidate, bars, start, end, model=None, embargo=True):\n    x = features(bars); side = pd.Series(0,index=bars.index,dtype=int)\n    if candidate[\'family\'] == \'breakout\':\n        n = candidate[\'lookback\']\n        side.loc[bars.close > bars.high.rolling(n).max().shift(1)] = 1\n        side.loc[bars.close < bars.low.rolling(n).min().shift(1)] = -1\n    elif candidate[\'family\'] == \'reversion\':\n        n = candidate[\'lookback\']; z = (bars.close-bars.close.rolling(n).mean())/bars.close.rolling(n).std()\n        quiet = x.trend.abs() < .75\n        side.loc[(z < -2) & quiet] = 1; side.loc[(z > 2) & quiet] = -1\n    elif model is not None and not isinstance(model,str):\n        good = x[FEATURES].notna().all(axis=1)\n        prob = pd.Series(np.nan,index=bars.index)\n        prob.loc[good] = model.predict_proba(x.loc[good,FEATURES])[:,1]\n        threshold = candidate[\'threshold\']\n        side.loc[prob > threshold] = 1; side.loc[prob < 1-threshold] = -1\n    # One fresh setup; no repeated re-entry into the same persistent signal.\n    side = side.where(side.ne(side.shift()),0)\n    s = pd.DataFrame({\'side\':side,\'distance\':x.atr*candidate[\'stop_atr\'],\'rr\':candidate[\'rr\']})\n    allowed = (s.index >= start) & (s.index < end) & (s.side != 0) & (s.distance > 0)\n    # Historical bars warm up causal features; no artificial removal of first test days.\n    allowed[:WARMUP_BARS] = False\n    return s.loc[allowed].dropna()\n\n\ndef empty_signals():\n    return pd.DataFrame(columns=[\'side\',\'distance\',\'rr\'], index=pd.DatetimeIndex([],tz=\'UTC\'))\n\n\ndef select(bars,ticks,train_start,val_start,val_end,cfg):\n    rows=[]; best=None; best_score=-np.inf\n    validation = ticks.loc[(ticks.index >= val_start) & (ticks.index < val_end)]\n    if len(validation)<2: raise ValueError(\'Empty validation period\')\n    for c in CANDIDATES:\n        model=fit_candidate(c,bars,train_start,val_start)\n        s=signals_for(c,bars,val_start,val_end,model)\n        r=backtest(validation,s,cfg)\n        ret=r[\'equity\'].iloc[-1]/cfg.initial_cash-1; dd=r[\'max_drawdown\']; count=len(r[\'trades\'])\n        eligible=count>=PROTOCOL[\'minimum_validation_trades\'] and ret>0 and dd<.08\n        score=ret/max(dd,.005) if eligible else -np.inf\n        rows.append({\'candidate\':c[\'name\'],\'return\':ret,\'drawdown\':dd,\'trades\':count,\n                     \'eligible\':eligible,\'ml_fit\': \'insufficient\' if isinstance(model,str) else (\'trained\' if model is not None else \'n/a\')})\n        if score>best_score: best_score=score; best=c\n    return best,pd.DataFrame(rows)\n\n\ndef research_run(ticks,bars,cfg=Execution(),mode=\'sample\'):\n    """Sample = smoke test. Full = 12m train/3m validation/3m OOS + final 6m."""\n    start=ticks.index[0]; end=ticks.index[-1]; selections=[]; combined=[]\n    if mode==\'sample\':\n        # Fixed session boundaries for the pinned sample, not tuned to returns.\n        train_start=pd.Timestamp(\'2016-04-04\',tz=\'America/New_York\').tz_convert(\'UTC\')\n        val_start=pd.Timestamp(\'2016-04-06\',tz=\'America/New_York\').tz_convert(\'UTC\')\n        hold_start=pd.Timestamp(\'2016-04-07\',tz=\'America/New_York\').tz_convert(\'UTC\')\n        chosen,table=select(bars,ticks,train_start,val_start,hold_start,cfg)\n        selections.append(table.assign(fold=\'smoke_validation\'))\n        oos_start=hold_start\n    else:\n        # First complete UTC month, exclude partial starting month.\n        month_start=start.normalize().replace(day=1)\n        if start>month_start: month_start += pd.DateOffset(months=1)\n        month_end=end.normalize().replace(day=1)\n        months=(month_end.year-month_start.year)*12+month_end.month-month_start.month\n        if months<24: raise ValueError(f\'Need >=24 complete calendar months, have {months}. No target claim possible.\')\n        # Refuse obvious missing weekday data, rather than silently treating gaps as flat markets.\n        dates=pd.date_range(month_start,month_end-pd.Timedelta(days=1),freq=\'B\',tz=\'UTC\')\n        observed=ticks.index.normalize().unique()\n        missing=dates.difference(observed)\n        if len(missing)/max(len(dates),1) > .03:\n            raise ValueError(f\'Missing {len(missing)} weekdays; inspect data coverage before research\')\n        ticks=ticks.loc[ticks.index<month_end]\n        end=month_end\n        hold_start=month_end-pd.DateOffset(months=6)\n        oos_start=month_start+pd.DateOffset(months=15)\n        cursor=oos_start\n        while cursor<hold_start:\n            test_end=min(cursor+pd.DateOffset(months=3),hold_start)\n            val_start=cursor-pd.DateOffset(months=3); train_start=val_start-pd.DateOffset(months=12)\n            c,table=select(bars,ticks,train_start,val_start,cursor,cfg)\n            selections.append(table.assign(fold=str(cursor),chosen=c[\'name\'] if c else \'CASH\'))\n            if c:\n                model=fit_candidate(c,bars,train_start,cursor)\n                combined.append(signals_for(c,bars,cursor,test_end,model))\n            cursor=test_end\n        val_start=hold_start-pd.DateOffset(months=3); train_start=val_start-pd.DateOffset(months=12)\n        chosen,table=select(bars,ticks,train_start,val_start,hold_start,cfg)\n        selections.append(table.assign(fold=\'final_selection\',chosen=chosen[\'name\'] if chosen else \'CASH\'))\n    # Freeze candidate before accessing final test outcomes. No holdout-driven retries.\n    model=fit_candidate(chosen,bars,train_start,hold_start) if chosen else None\n    final_signals=signals_for(chosen,bars,hold_start,end,model) if chosen else empty_signals()\n    frozen={\'candidate\':chosen or \'CASH\',\'protocol_hash\':protocol_hash(cfg), \'holdout_start\':str(hold_start)}\n    from pathlib import Path\n    model_state=None\n    if model is not None and not isinstance(model,str):\n        model_state={\'coef\':model[-1].coef_.tolist(),\'intercept\':model[-1].intercept_.tolist(),\n                     \'scaler_mean\':model[0].mean_.tolist(),\'scaler_scale\':model[0].scale_.tolist()}\n    frozen[\'model_state\']=model_state\n    frozen[\'selection_sha256\']=hashlib.sha256(json.dumps(frozen,sort_keys=True).encode()).hexdigest()\n    Path(\'artifacts\').mkdir(exist_ok=True)\n    Path(\'artifacts/frozen_selection.json\').write_text(json.dumps(frozen,indent=2))\n    # Data may already be loaded, but NO holdout outcome feeds selection, refit or this hash.\n    final_ticks=ticks.loc[ticks.index>=hold_start]\n    final=backtest(final_ticks,final_signals,cfg)\n    pre=None\n    if mode!=\'sample\':\n        pre_ticks=ticks.loc[(ticks.index>=oos_start)&(ticks.index<hold_start)]\n        pre_s=pd.concat(combined).sort_index() if combined else empty_signals()\n        pre=backtest(pre_ticks,pre_s,cfg)\n    stresses={}\n    for name,stress_cfg in {\n        \'base\':cfg,\n        \'spread_1.5_slip_2x\':replace(cfg,spread_multiplier=1.5,slippage=cfg.slippage*2),\n        \'spread_2_slip_3x_latency_1s\':replace(cfg,spread_multiplier=2.,slippage=cfg.slippage*3,latency_ms=1000),\n    }.items():\n        stresses[name]=backtest(final_ticks,final_signals,stress_cfg)\n    return {\'mode\':mode,\'selection\':pd.concat(selections,ignore_index=True), \'frozen\':frozen,\n            \'final\':final,\'walk_forward\':pre,\'stress\':stresses,\'final_signals\':final_signals}\n', encoding='utf-8')

from hippo_lab.research import research_run, CANDIDATES, PROTOCOL, protocol_hash, features, signals_for, fit_candidate
print('Protocol hash:', protocol_hash(execution))
print(json.dumps(PROTOCOL,indent=2,ensure_ascii=False))
display(pd.DataFrame(CANDIDATES))


## ۴ — بررسی علیت قبل از آزمایش

این تست کوچک **مصنوعی و فقط تست نرم‌افزار** است؛ به عنوان نتیجه مالی گزارش نمی‌شود. با دستکاری آینده، ویژگی‌ها و مدل آموزش‌دیده در گذشته نباید تغییر کنند. تست‌های بیشتر موتور در پوشه `tests` مخزن هستند.


In [ ]:
# @title ۴ — تست نشت اطلاعات و حسابداری
rng=np.random.default_rng(8056)
ix=pd.date_range('2020-01-01',periods=1800,freq='15min',tz='UTC')
close=1.2+np.cumsum(rng.normal(0,.0004,len(ix)))
fixture=pd.DataFrame({'open':close,'high':close+.0002,'low':close-.0002,'close':close,'spread':.0001},index=ix)
mutated=fixture.copy(); mutated.loc[ix[1300]:,'close']*=5
np.testing.assert_allclose(features(fixture).iloc[:1300],features(mutated).iloc[:1300],equal_nan=True)
m1=fit_candidate(CANDIDATES[4],fixture,ix[0],ix[1300]); m2=fit_candidate(CANDIDATES[4],mutated,ix[0],ix[1300])
assert not isinstance(m1,str)
np.testing.assert_allclose(m1[-1].coef_,m2[-1].coef_)
np.testing.assert_allclose(m1[0].mean_,m2[0].mean_)
check_index=pd.date_range('2024-01-02 14:00',periods=4,freq='s',tz='UTC')
check_ticks=pd.DataFrame({'bid':[1.2]*4,'ask':[1.2001]*4},index=check_index)
check_signals=pd.DataFrame({'side':[1],'distance':[.001],'rr':[1.]},index=check_index[:1])
check=backtest(check_ticks,check_signals,execution)
assert abs(check['trades'].pnl.sum()-(check['equity'].iloc[-1]-execution.initial_cash))<1e-7
assert check['trades'].entry_time.iloc[0]>check_index[0]
print('PASS: causal features, purged ML, executable spread + commission reconciliation')
del fixture,mutated,check_ticks,check_signals


## ۵ — اجرا و گزارش نهایی

آخرین سلول اجرای واقعی پژوهش و گزارش جامع را انجام می‌دهد. `sample` را با سود ماهانه مقایسه نکنید. تغییر پارامترها پس از مشاهده holdout آن را به داده اعتبارسنجی تبدیل می‌کند؛ برای آزمون بعدی تاریخ آینده جدید لازم است.

گزارش: تمام آزمایش‌های اعتبارسنجی، معاملات و R خالص، بازده ماه‌های کامل و ناقص، اکوییتی تیک‌به‌تیک، DD، گپ‌های داده، حسابداری هزینه، سناریوهای تنش ثابت، نمودار و بازنمونه‌گیری بلوکی فقط با تاریخچه کافی. تنش‌ها همان سیگنال‌های قفل‌شده را بازپخش می‌کنند؛ انتخاب مجدد انجام نمی‌شود. ارزیابی بازار دیگر باید **بدون انتخاب پس از دیدن نتیجه** و با ارز حساب/هزینه صحیح جداگانه اجرا شود.

افت سرمایه تاریخی و bootstrap تضمین افت آینده نیست. این بک‌تستر شبیه‌سازی quote-level است، نه اثبات قابلیت اجرای سفارش در حجم دلخواه. تاریخچه چندساله در حافظه نگهداری می‌شود؛ برای یک جفت ارز پرحجم از Colab High-RAM استفاده کنید. داده خام در Git ذخیره نمی‌شود.


In [ ]:
# @title ۵ — بک‌تست تیک‌به‌تیک، آزمون نهایی و گزارش
(PACKAGE / 'report.py').write_text('"""Auditable metrics; partial months never count toward the monthly objective."""\nfrom pathlib import Path\nimport html, json, base64\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom .research import PROTOCOL\n\n\ndef metrics(result):\n    e=result[\'equity\']; cash=result[\'config\'][\'initial_cash\']; t=result[\'trades\']\n    # Avoid duplicate timestamp reindexing but preserve full quote-level DD from engine.\n    e=e.groupby(level=0).last()\n    daily=e.resample(\'1D\').last().ffill()\n    daily_returns=daily.pct_change(fill_method=None); daily_returns.iloc[0]=daily.iloc[0]/cash-1\n    first=e.index[0]; last=e.index[-1]\n    month_end=e.resample(\'ME\').last()\n    rows=[]; prev=cash\n    for timestamp,value in month_end.items():\n        start=timestamp.normalize().replace(day=1)\n        next_start=start+pd.DateOffset(months=1)\n        # Data must bracket a whole UTC month. Missing weekends are allowed up to 3 days.\n        complete=first<=start+pd.Timedelta(days=3) and last>=next_start-pd.Timedelta(days=3)\n        rows.append({\'month\':str(start.date())[:7], \'return\':value/prev-1, \'complete\':complete})\n        prev=value\n    monthly=pd.DataFrame(rows)\n    pnl=t.pnl if len(t) else pd.Series(dtype=float)\n    wins=pnl[pnl>0].sum(); losses=-pnl[pnl<0].sum()\n    stats={\'total_return\':float(e.iloc[-1]/cash-1), \'max_drawdown\':result[\'max_drawdown\'],\n           \'trades\':len(t),\'win_rate\':float((pnl>0).mean()) if len(t) else None,\n           \'profit_factor\':float(wins/losses) if losses>0 else None,\n           \'net_pnl\':float(e.iloc[-1]-cash), \'mean_trade_pnl\':float(pnl.mean()) if len(t) else None,\n           \'gap_affected_trades\':int(t.quote_gap_over_60s.sum()) if len(t) else 0,\n           \'gap_affected_pnl\':float(t.loc[t.quote_gap_over_60s,\'pnl\'].sum()) if len(t) else 0.,\n           \'worst_realized_R\':float(t.realized_R.min()) if len(t) else None,\n           \'complete_months\':int(monthly.complete.sum()),\'drawdown_breaker_triggered\':result[\'halted\'],\n           \'breaker_overshoot\':max(0.,result[\'max_drawdown\']-result[\'config\'][\'drawdown_breaker\']),\n           \'submitted_signals\':result[\'submitted_signals\'], \'stale_or_outside_signals\':result[\'stale_or_outside_signals\']}\n    full=monthly.loc[monthly.complete,\'return\']\n    stats.update(mean_complete_month_return=float(full.mean()) if len(full) else None,\n                 fraction_months_ge_3pct=float((full>=.03).mean()) if len(full) else None,\n                 worst_month=float(full.min()) if len(full) else None)\n    # Bootstrap contiguous blocks only with enough days, not a week-long smoke test.\n    bootstrap=None\n    if len(daily_returns)>=180:\n        x=daily_returns.to_numpy(); rng=np.random.default_rng(8056); samples=[]; dds=[]\n        for _ in range(1000):\n            starts=rng.integers(0,len(x)-20+1,size=int(np.ceil(len(x)/20)))\n            r=np.concatenate([x[j:j+20] for j in starts])[:len(x)]\n            curve=np.r_[1.,np.cumprod(1+r)]\n            samples.append(curve[-1]**(30/len(r))-1)\n            dds.append(np.max(1-curve/np.maximum.accumulate(curve)))\n        bootstrap={\'monthly_equivalent_mean_ci95\':np.quantile(samples,[.025,.975]).tolist(),\n                   \'dd_p95\':float(np.quantile(dds,.95)),\n                   \'note\':\'Conditional 20-calendar-day block resampling; not a guarantee or selection-bias correction.\'}\n    return stats,monthly,daily_returns,bootstrap\n\n\ndef write_report(run,audit,folder=\'artifacts\',diagnostics=None):\n    folder=Path(folder); folder.mkdir(parents=True,exist_ok=True)\n    result=run[\'final\']; stats,monthly,daily,bootstrap=metrics(result)\n    no_selection=run[\'frozen\'][\'candidate\']==\'CASH\'\n    enough=run[\'mode\']!=\'sample\' and stats[\'trades\']>=30 and stats[\'complete_months\']>=5\n    status=\'NO_SELECTION\' if no_selection and run[\'mode\']!=\'sample\' else (\'INSUFFICIENT_EVIDENCE\' if not enough else \'EVALUATED_NOT_GUARANTEED\')\n    # Walk-forward and final holdout are separate accounts, not an invented stitched equity curve.\n    wf_stats=None\n    if run[\'walk_forward\'] is not None:\n        wf_stats=metrics(run[\'walk_forward\'])[0]\n    stresses={k:metrics(v)[0] for k,v in run[\'stress\'].items()}\n    forward_folds=max(0,run[\'selection\'].fold.nunique()-1) if run[\'mode\']!=\'sample\' else 0\n    sufficient_folds=forward_folds>=PROTOCOL[\'minimum_forward_folds\']\n    point_met=bool(enough and not result[\'halted\'] and stats[\'mean_complete_month_return\']>=.03 and stats[\'max_drawdown\']<.10 and all(s[\'total_return\']>0 and not s[\'drawdown_breaker_triggered\'] for s in stresses.values()))\n    summary={\'status\':status, \'point_estimate_met_unconfirmed\':point_met,\n             \'forward_folds\':forward_folds, \'sufficient_forward_folds\':sufficient_folds,\n             \'long_term_target_verified\':False, \'input_sufficient_for_full_protocol\':run[\'mode\']!=\'sample\',\n             \'no_selection\':no_selection, \'holdout_evidence_sufficient_for_initial_review\':enough,\n             \'monthly_goal_definition\':\'Mean net complete calendar-month return >=3%; not a promise of 3% every month. Long-term target requires >=24 OOS months.\',\'objective\':\'3–4% monthly, maximum equity drawdown <10%; not guaranteed\',\n             \'data_audit\':audit,\'frozen_selection\':run[\'frozen\'],\'final_holdout\':stats,\n             \'walk_forward_separate_account\':wf_stats,\'stress_same_frozen_signals\':stresses,\n             \'bootstrap\':bootstrap, \'limitations\':[\n                 \'Public mirror quotes not independently broker-certified; no exchange order book or market impact.\',\n                 \'Quote-currency account only (GBPUSD: USD). Other pairs require their quote-currency account; no cross-FX conversion.\',\n                 \'Single position, intraday NY liquidation; no swaps; observed quote gaps can overshoot risk controls.\',\n                 \'Best-of-six selection is biased; holdout is not used for tuning. Revisiting after results voids its untouched status.\',\n                 \'One week cannot establish profitability or transfer across markets; >=24 input months is an exploratory run, not 24 OOS months.\',\n                 \'Validation reject -> no deployment, not evidence that cash is superior.\']}\n    (folder/\'summary.json\').write_text(json.dumps(summary,indent=2,ensure_ascii=False,allow_nan=False))\n    run[\'selection\'].to_csv(folder/\'validation_all_candidates.csv\',index=False)\n    result[\'trades\'].to_csv(folder/\'holdout_trades.csv\',index=False)\n    monthly.to_csv(folder/\'holdout_months.csv\',index=False)\n    result[\'equity\'].resample(\'1min\').last().to_csv(folder/\'equity_1min.csv\')\n    # Full tick equity remains in memory; optional parquet audit, not committed.\n    result[\'equity\'].to_frame().to_parquet(folder/\'equity_every_tick.parquet\')\n    if diagnostics is not None: diagnostics.to_csv(folder/\'smoke_unselected_candidates.csv\',index=False)\n    fig,axes=plt.subplots(2,1,figsize=(11,6),sharex=True)\n    plot_e=result[\'equity\'].groupby(level=0).last()\n    axes[0].plot(plot_e.index,plot_e.values); axes[0].set_ylabel(\'Liquidation equity\')\n    peak=plot_e.cummax().clip(lower=result[\'config\'][\'initial_cash\'])\n    axes[1].plot(plot_e.index,100*(1-plot_e/peak)); axes[1].set_ylabel(\'Drawdown %\')\n    fig.suptitle(status+\' — frozen holdout (CASH may reflect inadequate evidence)\')\n    fig.tight_layout(); fig.savefig(folder/\'equity.png\'); plt.close(fig)\n    def table(df): return df.to_html(index=False,escape=True,float_format=lambda x:f\'{x:.6f}\')\n    body=f\'\'\'<!doctype html><html lang="fa" dir="rtl"><meta charset="utf-8"><title>HIPO audit</title>\n    <style>body{{font-family:system-ui;max-width:1100px;margin:32px auto;padding:20px;background:#f5f7fb;color:#17233a}}table{{direction:ltr;border-collapse:collapse;width:100%;background:white}}td,th{{padding:8px;border:1px solid #ccd}}pre{{direction:ltr;text-align:left;white-space:pre-wrap}}img{{width:100%}}</style>\n    <h1>گزارش پژوهش تیک\u200cبه\u200cتیک HIPO</h1><h2>{status}</h2>\n    <p>این گزارش تضمین سود نیست. داده کوتاه فقط آزمون مهندسی است؛ عدم معامله، تأیید استراتژی نیست.</p>\n    <img src="data:image/png;base64,{base64.b64encode((folder/\'equity.png\').read_bytes()).decode()}"><h2>تمام آزمایش\u200cهای اعتبارسنجی</h2>{table(run[\'selection\'])}\n    <h2>ماه\u200cها (ناقص\u200cها خارج از هدف)</h2>{table(monthly)}\n    <h2>اجرای تشخیصی فرضیه\u200cهای از پیش تعیین\u200cشده — نه انتخاب و نه اثبات سود</h2>{table(diagnostics) if diagnostics is not None else \'\'}\n    <h2>فرض\u200cها، داده و تنش هزینه</h2><pre>{html.escape(json.dumps(summary,indent=2,ensure_ascii=False))}</pre></html>\'\'\'\n    (folder/\'report.html\').write_text(body,encoding=\'utf-8\')\n    return summary\n', encoding='utf-8')

from hippo_lab.report import write_report, metrics
mode = 'sample' if DATA_MODE == 'sample' else 'full'
run = research_run(ticks,bars,execution,mode=mode)
diagnostics = None
if mode == 'sample':
    rows=[]
    for candidate in CANDIDATES:
        model=fit_candidate(candidate,bars,ticks.index[0],pd.Timestamp('2016-04-07',tz='America/New_York').tz_convert('UTC'))
        signals=signals_for(candidate,bars,ticks.index[0],ticks.index[-1],model)
        diagnostic=backtest(ticks,signals,execution)
        rows.append({'candidate':candidate['name'],'role':'SMOKE ONLY — NOT OOS',**metrics(diagnostic)[0]})
    diagnostics=pd.DataFrame(rows)
summary=write_report(run,data_audit,folder='artifacts',diagnostics=diagnostics)
print(json.dumps(summary,indent=2,ensure_ascii=False))
display(run['selection'])
if diagnostics is not None: display(diagnostics)
display(HTML(Path('artifacts/report.html').read_text()))
print('Files: artifacts/report.html, summary.json, holdout_trades.csv, holdout_months.csv, equity_every_tick.parquet')
# Optional: download a compact report ZIP; tick equity can be very large, so exclude it.
import zipfile
with zipfile.ZipFile('HIPO_report.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in Path('artifacts').glob('*'):
        if p.is_file() and p.suffix in ['.json','.html','.png','.csv']: z.write(p,arcname=p.name)
if 'google.colab' in sys.modules:
    from google.colab import files
    display(HTML('<p>گزارش در HIPO_report.zip آماده است؛ برای دریافت: files.download("HIPO_report.zip")</p>'))
